# Fold-Local Rebound Priority Conflict v1

This notebook checks whether the new fold-local joint reward targets can safely feed weighted BC/PPO. The specific risk is overfit through a target-label conflict: rebound rows are supposed to teach re-entry, but the current target-position rule gives downtrend risk caps priority over rebound floors.

## Experiment Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- **Direct comparison baseline**: `fold_local_joint_reward_targets_v1` and the earlier `state_objective_state_conditioned_loss_v1` overfit screen.
- **Role**: diagnose, not promotion.
- **Changed layer**: `model_selection`, `reward`, `sampling`.
- **Target window**: fold-local train rows used before any fixed-split/walk-forward PPO run.
- **Acceptance gate**: do not start BC/PPO unless target rows contain effective rebound lift, no eval overlap, and a state head or weighting rule avoids high false-positive release.
- **Failure mode**: if downtrend caps erase rebound lift or a release head fires broadly outside the rebound window, return to target semantics instead of training.
- **Comparison-table row key**: `fold_local_rebound_priority_conflict_v1`.

Historical context:

- Closest success: `fold_local_joint_reward_targets_v1` removed eval-overlap leakage and selected `risk_budget_usage_v1` as a healthy action surface.
- Closest failure: `state_objective_state_conditioned_loss_v1` fit rebound rows in-sample but failed holdout, a pattern consistent with sparse-label memorization.
- Why this is not a duplicate: this notebook audits whether the generated target labels actually encode rebound participation before another BC/PPO branch is opened.


## External Basis

- [Qlib GitHub](https://github.com/microsoft/qlib) and [Qlib arXiv 2009.11189](https://arxiv.org/abs/2009.11189): keep data, model, backtest, and analysis inside a reproducible workflow.
- [FinRL GitHub](https://github.com/AI4Finance-Foundation/FinRL) and [FinRL arXiv 2011.09607](https://arxiv.org/abs/2011.09607): trading RL should keep data processing, environment, training, and evaluation explicit.
- [Focal Loss](https://arxiv.org/abs/1708.02002): sparse/hard states need explicit imbalance handling, but imbalance handling cannot fix a target that encodes the wrong action.
- [Prioritized Experience Replay](https://arxiv.org/abs/1511.05952): high-error/high-value states can be prioritized, but only after the target semantics are valid.
- [Probability of Backtest Overfitting](https://doi.org/10.21314/JCF.2016.322): do not launch another model branch from a threshold screen without leakage and holdout guards.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from scripts.evaluate_factor_pack import _load_ohlcv_frames
from src.soxl_quant.app.representation_diagnostics import (
    StateHeadTargetRecipeConfig,
    score_binary_representation_feature_sets,
    score_state_head_target_recipes,
)
from src.soxl_quant.data import engineer_feature_frame
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

CARD = ROOT / 'runtime/model_selection/card/fold_local_rebound_priority_conflict_v1'
SOURCE_CARD = ROOT / 'runtime/model_selection/card/fold_local_joint_reward_targets_v1'
TARGETS_PATH = SOURCE_CARD / 'fold_local_joint_targets.csv'
CARD.mkdir(parents=True, exist_ok=True)

EXPERIMENT = 'fold_local_rebound_priority_conflict_v1'
CREATED_AT = datetime.now(timezone.utc).isoformat()

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Load Fold-Local Targets And Fresh Feature Columns

The target table comes from the latest leakage-guarded card. SOXX/SOXL spread and SOXX signal features are rejoined from the canonical panel so the state-head screen does not rely on zero-filled diagnostic columns in a prior path export.


In [2]:
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])

soxl, soxx = _load_ohlcv_frames()
features, _ = engineer_feature_frame(
    soxl,
    soxx,
    feature_packs=('transition_signals', 'soxx_signals', 'soxx_spread_oscillation'),
)
features = features.reset_index().rename(columns={'index': 'datetime'})
features['datetime'] = pd.to_datetime(features['datetime'])

feature_cols = [
    'mom_5',
    'mom_21',
    'drawdown_63',
    'realized_vol_21',
    'vol_of_vol_21',
    'REL_VOLUME_RATIO',
    'PRICE_RATIO_MOM',
    'SOXX_BIAS_10',
    'SIGNED_TE_REVERSION_10',
    'TRANS_TE_ZSCORE',
    'SOXL_SOXX_RATIO_Z_252',
    'SOXL_SOXX_RATIO_WIDEN_5',
    'SOXL_SOXX_SPREAD_VOL_Z_21',
    'SOXL_SOXX_CHOP_DIVERGENCE_21',
]
feature_frame = features[['datetime', *feature_cols]].copy()

merge_base = targets.drop(columns=[c for c in feature_cols if c in targets.columns], errors='ignore')
frame = merge_base.merge(feature_frame, on='datetime', how='left')
for col in ['is_rebound_cluster', 'is_downtrend_state']:
    frame[col] = frame[col].fillna(False).astype(bool)

fold_date_sets = {
    fold: set(group['datetime'].dt.normalize())
    for fold, group in frame.groupby('fold')
}
folds = sorted(fold_date_sets)
overlap_dates = set.intersection(*(fold_date_sets[f] for f in folds)) if len(folds) >= 2 else set()

coverage = {
    'rows': int(len(frame)),
    'folds': folds,
    'fold_overlap_dates': int(len(overlap_dates)),
    'source_target_scope_unique': sorted(frame['target_scope'].dropna().astype(str).unique().tolist()),
    'date_start': str(frame['datetime'].min().date()),
    'date_end': str(frame['datetime'].max().date()),
}
coverage


{'rows': 1261,
 'folds': ['wf_2022_train', 'wf_2023_2024_train'],
 'fold_overlap_dates': 505,
 'source_target_scope_unique': ['fold_local'],
 'date_start': '2020-01-02',
 'date_end': '2022-12-30'}

## Target Priority Conflict

Current shared logic deliberately chooses the downtrend risk cap before rebound participation. This is safe for drawdown, but it means a rebound label can be present while the actual BC target remains the same low-risk cap. The table below quantifies the conflict.


In [3]:
frame['current_target_position'] = [
    state_objective_target_position_from_row(row)
    for row in frame.to_dict('records')
]
frame['no_rebound_position'] = np.where(
    frame['is_downtrend_state'],
    frame['risk_target_position'],
    frame['teacher_position'],
)
frame['candidate_release_position'] = np.where(
    frame['is_downtrend_state'],
    np.maximum(frame['risk_target_position'], frame['rebound_target_position']),
    np.maximum(frame['teacher_position'], frame['rebound_target_position']),
)
frame['true_release_position'] = np.where(
    frame['is_rebound_cluster'],
    frame['candidate_release_position'],
    frame['current_target_position'],
)
frame['current_effective_lift'] = frame['current_target_position'] - frame['no_rebound_position']
frame['candidate_release_lift'] = frame['candidate_release_position'] - frame['no_rebound_position']
frame['true_release_lift'] = frame['true_release_position'] - frame['no_rebound_position']

priority_conflict = (
    frame.groupby(['fold', 'is_rebound_cluster', 'is_downtrend_state'], dropna=False)
    .agg(
        rows=('datetime', 'count'),
        mean_teacher_position=('teacher_position', 'mean'),
        mean_risk_target_position=('risk_target_position', 'mean'),
        mean_rebound_target_position=('rebound_target_position', 'mean'),
        mean_current_target_position=('current_target_position', 'mean'),
        mean_current_effective_lift=('current_effective_lift', 'mean'),
        mean_candidate_release_lift=('candidate_release_lift', 'mean'),
        positive_current_lift_rows=('current_effective_lift', lambda s: int((s.abs() > 1e-9).sum())),
        positive_release_lift_rows=('true_release_lift', lambda s: int((s.abs() > 1e-9).sum())),
    )
    .reset_index()
)
priority_conflict.to_csv(CARD / 'target_priority_conflict.csv', index=False)
priority_conflict


,fold,is_rebound_cluster,is_downtrend_state,rows,mean_teacher_position,mean_risk_target_position,mean_rebound_target_position,mean_current_target_position,mean_current_effective_lift,mean_candidate_release_lift,positive_current_lift_rows,positive_release_lift_rows
0,wf_2022_train,False,False,262,0.426693,0.426693,0.426693,0.426693,0.0,0.000000,0,0
1,wf_2022_train,False,True,177,0.214033,0.178081,0.214033,0.178081,0.0,0.035952,0,0
2,wf_2022_train,True,False,9,0.574267,0.574267,0.574267,0.574267,0.0,0.000000,0,0
3,wf_2022_train,True,True,57,0.552643,0.223684,0.605274,0.223684,0.0,0.381590,0,57
4,wf_2023_2024_train,False,False,269,0.421220,0.421220,0.421220,0.421220,0.0,0.000000,0,0
5,wf_2023_2024_train,False,True,421,0.166341,0.151225,0.166341,0.151225,0.0,0.015115,0,0
6,wf_2023_2024_train,True,False,9,0.574267,0.574267,0.574267,0.574267,0.0,0.000000,0,0
7,wf_2023_2024_train,True,True,57,0.552643,0.223684,0.605274,0.223684,0.0,0.381590,0,57


In [4]:
conflict_summary = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'rows': int(len(frame)),
    'rebound_rows': int(frame['is_rebound_cluster'].sum()),
    'downtrend_rows': int(frame['is_downtrend_state'].sum()),
    'rebound_downtrend_rows': int((frame['is_rebound_cluster'] & frame['is_downtrend_state']).sum()),
    'rebound_not_downtrend_rows': int((frame['is_rebound_cluster'] & ~frame['is_downtrend_state']).sum()),
    'current_positive_rebound_lift_rows': int((frame['is_rebound_cluster'] & (frame['current_effective_lift'].abs() > 1e-9)).sum()),
    'candidate_release_positive_lift_rows': int((frame['is_rebound_cluster'] & (frame['true_release_lift'].abs() > 1e-9)).sum()),
    'mean_candidate_release_lift_on_rebound_downtrend': float(
        frame.loc[frame['is_rebound_cluster'] & frame['is_downtrend_state'], 'candidate_release_lift'].mean()
    ),
    'fold_overlap_dates': int(len(overlap_dates)),
    'fold_overlap_blocks_direct_fold_heldout_claims': bool(len(overlap_dates) > 0),
}
write_json(CARD / 'conflict_summary.json', conflict_summary)
conflict_summary


{'experiment': 'fold_local_rebound_priority_conflict_v1',
 'created_at': '2026-05-25T17:22:06.085151+00:00',
 'source_targets': 'runtime/model_selection/card/fold_local_joint_reward_targets_v1/fold_local_joint_targets.csv',
 'rows': 1261,
 'rebound_rows': 132,
 'downtrend_rows': 712,
 'rebound_downtrend_rows': 114,
 'rebound_not_downtrend_rows': 18,
 'current_positive_rebound_lift_rows': 0,
 'candidate_release_positive_lift_rows': 114,
 'mean_candidate_release_lift_on_rebound_downtrend': 0.38159001504193146,
 'fold_overlap_dates': 505,
 'fold_overlap_blocks_direct_fold_heldout_claims': True}

## Non-Overlapping Event-Fold State-Head Screen

The two fold-local train sets overlap on 2020-2021 dates, so a naive `fold` holdout would leak the same dates across train and eval. For this diagnostic, only the non-overlapping 2020 crash/recovery event window from `wf_2022_train` is used, split into `2020Q1` and `2020Q2`. This is a small event-screen, not a promotion backtest.


In [5]:
event_frame = frame[
    (frame['fold'].astype(str) == 'wf_2022_train')
    & (frame['datetime'] >= pd.Timestamp('2020-01-01'))
    & (frame['datetime'] <= pd.Timestamp('2020-06-30'))
].copy()
event_frame['event_fold'] = event_frame['datetime'].dt.to_period('Q').astype(str)

event_summary = (
    event_frame.groupby('event_fold')
    .agg(
        rows=('datetime', 'count'),
        rebound_rows=('is_rebound_cluster', 'sum'),
        downtrend_rows=('is_downtrend_state', 'sum'),
        release_lift_rows=('true_release_lift', lambda s: int((s.abs() > 1e-9).sum())),
        start=('datetime', 'min'),
        end=('datetime', 'max'),
    )
    .reset_index()
)
event_summary.to_csv(CARD / 'event_fold_summary.csv', index=False)
event_summary


,event_fold,rows,rebound_rows,downtrend_rows,release_lift_rows,start,end
0,2020Q1,62,10,35,10,2020-01-02,2020-03-31
1,2020Q2,63,56,50,47,2020-04-01,2020-06-30


In [6]:
feature_sets = {
    'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63'],
    'momentum_soxx_signals': ['mom_5', 'mom_21', 'drawdown_63', 'REL_VOLUME_RATIO', 'PRICE_RATIO_MOM', 'SOXX_BIAS_10'],
    'signed_te_combo': ['mom_5', 'mom_21', 'drawdown_63', 'SIGNED_TE_REVERSION_10', 'TRANS_TE_ZSCORE'],
    'soxx_spread_oscillation': [
        'SOXL_SOXX_RATIO_Z_252',
        'SOXL_SOXX_RATIO_WIDEN_5',
        'SOXL_SOXX_SPREAD_VOL_Z_21',
        'SOXL_SOXX_CHOP_DIVERGENCE_21',
    ],
    'all_candidate': [
        'mom_5',
        'mom_21',
        'drawdown_63',
        'realized_vol_21',
        'vol_of_vol_21',
        'REL_VOLUME_RATIO',
        'PRICE_RATIO_MOM',
        'SOXX_BIAS_10',
        'SIGNED_TE_REVERSION_10',
        'TRANS_TE_ZSCORE',
        'SOXL_SOXX_RATIO_Z_252',
        'SOXL_SOXX_RATIO_WIDEN_5',
        'SOXL_SOXX_SPREAD_VOL_Z_21',
        'SOXL_SOXX_CHOP_DIVERGENCE_21',
    ],
}

representation = score_binary_representation_feature_sets(
    event_frame,
    label_col='is_rebound_cluster',
    fold_col='event_fold',
    feature_sets=feature_sets,
    min_positive_train=5,
    min_positive_eval=5,
    min_holdout_auc=0.60,
    min_holdout_accuracy=0.55,
)
representation.feature_set_scores.to_csv(CARD / 'representation_feature_set_scores.csv', index=False)
representation.fold_metrics.to_csv(CARD / 'representation_fold_metrics.csv', index=False)
representation.feature_effects.to_csv(CARD / 'representation_feature_effects.csv', index=False)
representation.feature_set_scores


,feature_set,features,feature_count,folds,covered_folds,coverage_gate,holdout_auc_min,holdout_auc_mean,holdout_accuracy_min,holdout_accuracy_mean,representation_gate
0,signed_te_combo,"mom_5,mom_21,drawdown_63,SIGNED_TE_REVERSION_1...",5,2,2,True,0.975000,0.978571,0.158730,0.321301,False
1,momentum_drawdown,"mom_5,mom_21,drawdown_63",3,2,2,True,0.967308,0.972174,0.222222,0.562724,False
2,soxx_spread_oscillation,"SOXL_SOXX_RATIO_Z_252,SOXL_SOXX_RATIO_WIDEN_5,...",4,2,2,True,0.951531,0.974804,0.269841,0.554275,False
3,all_candidate,"mom_5,mom_21,drawdown_63,realized_vol_21,vol_o...",14,2,2,True,0.943878,0.969054,0.222222,0.498208,False
4,momentum_soxx_signals,"mom_5,mom_21,drawdown_63,REL_VOLUME_RATIO,PRIC...",6,2,2,True,0.767308,0.828807,0.174603,0.498592,False


In [7]:
recipe = score_state_head_target_recipes(
    event_frame,
    label_col='is_rebound_cluster',
    fold_col='event_fold',
    true_target_col='true_release_position',
    default_target_col='no_rebound_position',
    state_target_col='candidate_release_position',
    feature_sets=feature_sets,
    thresholds=(-1.0, -0.5, 0.0, 0.5, 1.0),
    min_positive_train=5,
    min_positive_eval=5,
    config=StateHeadTargetRecipeConfig(
        min_recall=0.60,
        min_precision=0.40,
        max_false_positive_rate=0.25,
        max_effective_false_positive_rate=0.10,
        max_false_positive_lift_mean=0.02,
        max_false_positive_lift_max=0.20,
        max_target_mae=0.08,
        max_rebound_mae=0.08,
        min_covered_folds=2,
    ),
)
recipe.recipe_scores.to_csv(CARD / 'state_head_recipe_scores.csv', index=False)
recipe.row_scores.to_csv(CARD / 'state_head_row_scores.csv', index=False)
recipe.recipe_scores.head(15)


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,...,false_positive_rate,effective_false_positive,effective_false_positive_rate,false_positive_lift_mean,false_positive_lift_max,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss
0,momentum_drawdown,-1.0,125,66,59,2,44,29,22,30,...,0.491525,0,0.0,0.0,0.0,0.042534,0.080557,0.0,False,0.483608
1,soxx_spread_oscillation,-1.0,125,66,59,2,49,34,17,25,...,0.576271,0,0.0,0.0,0.0,0.021627,0.040960,0.0,False,0.652542
2,soxx_spread_oscillation,-0.5,125,66,59,2,31,17,35,42,...,0.288136,0,0.0,0.0,0.0,0.098259,0.186097,0.0,False,0.721840
3,soxx_spread_oscillation,0.0,125,66,59,2,20,10,46,49,...,0.169492,0,0.0,0.0,0.0,0.132548,0.251039,0.0,False,1.411466
4,momentum_drawdown,-0.5,125,66,59,2,20,16,46,43,...,0.271186,0,0.0,0.0,0.0,0.131733,0.249494,0.0,False,1.451478
5,soxx_spread_oscillation,0.5,125,66,59,2,17,1,49,58,...,0.016949,0,0.0,0.0,0.0,0.142244,0.269402,0.0,False,1.621343
6,all_candidate,0.0,125,66,59,2,17,14,49,45,...,0.237288,0,0.0,0.0,0.0,0.143764,0.272281,0.0,False,1.625743
7,all_candidate,0.5,125,66,59,2,16,4,50,55,...,0.067797,0,0.0,0.0,0.0,0.146885,0.278191,0.0,False,1.695379
8,momentum_drawdown,0.0,125,66,59,2,15,4,51,55,...,0.067797,0,0.0,0.0,0.0,0.146964,0.278342,0.0,False,1.756215
9,momentum_soxx_signals,-1.0,125,66,59,2,28,42,38,17,...,0.711864,0,0.0,0.0,0.0,0.109738,0.207838,0.0,False,1.784336


## Full-Period False-Positive Audit For The Best Event Recipe

The event-fold recipe ranks `momentum_drawdown` with a permissive threshold as the closest candidate, but permissive thresholds can over-release in later non-rebound periods. This audit trains the same centroid head on the 2020H1 event window and applies it to the full `wf_2022_train` target span. This is deliberately conservative: a candidate that fires broadly outside the event should not become a PPO warm start.


In [8]:
best_recipe = recipe.recipe_scores.iloc[0].to_dict() if not recipe.recipe_scores.empty else {}
best_feature_set = str(best_recipe.get('feature_set', ''))
best_threshold = float(best_recipe.get('threshold', 0.0))
best_features = feature_sets.get(best_feature_set, [])

if not best_features:
    false_positive_audit = pd.DataFrame()
else:
    train = event_frame.copy()
    train_values = train[best_features].apply(pd.to_numeric, errors='coerce')
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0, ddof=0).replace(0.0, 1.0).fillna(1.0)
    train_z = ((train_values - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    labels = train['is_rebound_cluster'].to_numpy(dtype=bool)
    pos_center = train_z[labels].mean(axis=0)
    neg_center = train_z[~labels].mean(axis=0)

    scored = frame[frame['fold'].astype(str) == 'wf_2022_train'].copy()
    eval_values = scored[best_features].apply(pd.to_numeric, errors='coerce')
    eval_z = ((eval_values - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    scored['state_score'] = np.linalg.norm(eval_z - neg_center, axis=1) - np.linalg.norm(eval_z - pos_center, axis=1)
    scored['predicted_release'] = scored['state_score'] >= best_threshold
    scored['period'] = np.where(scored['datetime'] <= pd.Timestamp('2020-06-30'), 'event_2020h1', 'post_event')

    audit_rows = []
    for period, group in scored.groupby('period', sort=True):
        labels = group['is_rebound_cluster'].astype(bool)
        preds = group['predicted_release'].astype(bool)
        negatives = ~labels
        tp = int((labels & preds).sum())
        fp = int((negatives & preds).sum())
        fn = int((labels & ~preds).sum())
        tn = int((negatives & ~preds).sum())
        effective_fp = int((negatives & preds & (group['candidate_release_lift'].abs() > 0.01)).sum())
        audit_rows.append({
            'period': period,
            'rows': int(len(group)),
            'true_positive': tp,
            'false_positive': fp,
            'false_negative': fn,
            'true_negative': tn,
            'false_positive_rate': fp / max(fp + tn, 1),
            'effective_false_positive_rows': effective_fp,
            'prediction_rate': float(preds.mean()) if len(group) else 0.0,
            'mean_candidate_lift_on_effective_fp': float(
                group.loc[negatives & preds & (group['candidate_release_lift'].abs() > 0.01), 'candidate_release_lift'].mean()
            ) if effective_fp else 0.0,
        })
    false_positive_audit = pd.DataFrame(audit_rows)

false_positive_audit.to_csv(CARD / 'full_period_false_positive_audit.csv', index=False)
false_positive_audit


,period,rows,true_positive,false_positive,false_negative,true_negative,false_positive_rate,effective_false_positive_rows,prediction_rate,mean_candidate_lift_on_effective_fp
0,event_2020h1,125,66,48,0,11,0.813559,2,0.912000,0.181427
1,post_event,380,0,303,0,77,0.797368,25,0.797368,0.185320


## Decision

The generated sidecars are machine-readable; the conclusion below is the run-level decision for this notebook.


In [9]:
best_representation = representation.feature_set_scores.iloc[0].to_dict() if not representation.feature_set_scores.empty else {}
best_recipe = recipe.recipe_scores.iloc[0].to_dict() if not recipe.recipe_scores.empty else {}
post_event_fp_rate = float(
    false_positive_audit.loc[false_positive_audit['period'] == 'post_event', 'false_positive_rate'].iloc[0]
) if not false_positive_audit.empty and (false_positive_audit['period'] == 'post_event').any() else float('nan')
post_event_effective_fp = int(
    false_positive_audit.loc[false_positive_audit['period'] == 'post_event', 'effective_false_positive_rows'].iloc[0]
) if not false_positive_audit.empty and (false_positive_audit['period'] == 'post_event').any() else 0

current_lift_rows = int(conflict_summary['current_positive_rebound_lift_rows'])
release_lift_rows = int(conflict_summary['candidate_release_positive_lift_rows'])
recipe_gate = bool(best_recipe.get('recipe_gate', False))
false_positive_gate = bool(np.isfinite(post_event_fp_rate) and post_event_fp_rate <= 0.25 and post_event_effective_fp == 0)
target_semantics_gate = bool(current_lift_rows > 0)
release_candidate_gate = bool(release_lift_rows > 0 and recipe_gate and false_positive_gate)

if not target_semantics_gate:
    decision = 'rebound_priority_conflict_blocks_bc_ppo'
    next_mechanism = 'revise_state_objective_target_semantics_before_weighted_bc'
elif not release_candidate_gate:
    decision = 'release_state_head_not_ready_for_bc_ppo'
    next_mechanism = 'screen_safer_release_target_or_drawdown_controller'
else:
    decision = 'release_state_head_recipe_ready_for_bc_screen'
    next_mechanism = 'convert_release_recipe_to_fold_local_weighted_bc_then_walkforward'

selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'fold_local_joint_reward_targets_v1',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'reward', 'sampling'],
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'target_rows': int(len(frame)),
    'fold_overlap_dates': int(len(overlap_dates)),
    'rebound_rows': int(conflict_summary['rebound_rows']),
    'rebound_downtrend_rows': int(conflict_summary['rebound_downtrend_rows']),
    'current_positive_rebound_lift_rows': current_lift_rows,
    'candidate_release_positive_lift_rows': release_lift_rows,
    'best_representation_feature_set': str(best_representation.get('feature_set', '')),
    'best_representation_auc_min': float(best_representation.get('holdout_auc_min', np.nan)),
    'best_representation_accuracy_min': float(best_representation.get('holdout_accuracy_min', np.nan)),
    'best_recipe_feature_set': str(best_recipe.get('feature_set', '')),
    'best_recipe_threshold': float(best_recipe.get('threshold', np.nan)),
    'best_recipe_recall': float(best_recipe.get('state_recall', np.nan)),
    'best_recipe_precision': float(best_recipe.get('state_precision', np.nan)),
    'best_recipe_false_positive_rate': float(best_recipe.get('false_positive_rate', np.nan)),
    'best_recipe_gate': recipe_gate,
    'post_event_false_positive_rate': post_event_fp_rate,
    'post_event_effective_false_positive_rows': post_event_effective_fp,
    'target_semantics_gate': target_semantics_gate,
    'release_candidate_gate': release_candidate_gate,
    'training_allowed': False,
    'promotion_allowed': False,
    'promotion_blocker': 'This is a target-semantics and state-head diagnostic; no fixed-split or walk-forward PPO/backtest gate was run.',
    'acceptance_gate_for_next_run': {
        'current_target_must_encode_positive_rebound_lift': True,
        'fold_overlap_dates_must_not_be_used_as_holdout': True,
        'state_head_recipe_gate_required': True,
        'post_event_false_positive_rate_max': 0.25,
        'fixed_split_required': True,
        'walkforward_required': True,
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1511.05952',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'target_priority_conflict.csv',
        'conflict_summary.json',
        'event_fold_summary.csv',
        'representation_feature_set_scores.csv',
        'representation_fold_metrics.csv',
        'representation_feature_effects.csv',
        'state_head_recipe_scores.csv',
        'state_head_row_scores.csv',
        'full_period_false_positive_audit.csv',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'conclusion': [
        f"Current target logic leaves {current_lift_rows} rebound rows with positive effective lift, despite {conflict_summary['rebound_rows']} rebound-labeled rows.",
        f"A release-priority target would create lift on {release_lift_rows} rebound rows, but the best event state-head recipe gate is {recipe_gate}.",
        f"The best event feature set is {selection['best_recipe_feature_set']} with recall {selection['best_recipe_recall']:.3f}, precision {selection['best_recipe_precision']:.3f}, and event FPR {selection['best_recipe_false_positive_rate']:.3f}.",
        f"Post-event audit false-positive rate is {post_event_fp_rate:.3f}; therefore do not start weighted BC/PPO until target semantics and release gating are fixed.",
    ],
    'selection': selection,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

for line in summary['conclusion']:
    display(Markdown(f'- {line}'))
selection


- Current target logic leaves 0 rebound rows with positive effective lift, despite 132 rebound-labeled rows.

- A release-priority target would create lift on 114 rebound rows, but the best event state-head recipe gate is False.

- The best event feature set is momentum_drawdown with recall 0.667, precision 0.603, and event FPR 0.492.

- Post-event audit false-positive rate is 0.797; therefore do not start weighted BC/PPO until target semantics and release gating are fixed.

{'experiment': 'fold_local_rebound_priority_conflict_v1',
 'created_at': '2026-05-25T17:22:06.085151+00:00',
 'decision': 'rebound_priority_conflict_blocks_bc_ppo',
 'next_mechanism': 'revise_state_objective_target_semantics_before_weighted_bc',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'fold_local_joint_reward_targets_v1',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'reward', 'sampling'],
 'source_targets': 'runtime/model_selection/card/fold_local_joint_reward_targets_v1/fold_local_joint_targets.csv',
 'target_rows': 1261,
 'fold_overlap_dates': 505,
 'rebound_rows': 132,
 'rebound_downtrend_rows': 114,
 'current_positive_rebound_lift_rows': 0,
 'candidate_release_positive_lift_rows': 114,
 'best_representation_feature_set': 'signed_te_combo',
 'best_representation_auc_min': 0.975,
 'best_representation_accuracy_min': 0.15873015873015872,
 'best_recipe_feature_set': 'momentum_drawdown',
 'best_recipe_threshold': -1.0,